## Objective


Build a reproducible churn classification workflow that compares tabular ML models, prioritizes recall and ROC-AUC, creates local explainability outputs, and saves artifacts for the FastAPI application.


## Data loading


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
SAMPLE_PATH = ROOT / 'data' / 'sample' / 'sample_churn.csv'
MODEL_DIR = ROOT / 'backend' / 'models'
TARGET = 'Churn'

raw_files = sorted(RAW_DIR.glob('*.csv'))
data_path = raw_files[0] if raw_files else SAMPLE_PATH
df = pd.read_csv(data_path)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.head()

## Quick EDA


In [ ]:
print(df.shape)
df.describe(include='all').transpose().head(20)

## Target distribution


In [ ]:
df[TARGET].value_counts(normalize=True).rename('share').to_frame()

## Missing values


In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

## Numeric/categorical columns


In [ ]:
y = df[TARGET].map({'Yes': 1, 'No': 0}).astype(int)
X = df.drop(columns=[TARGET])
if 'customerID' in X.columns:
    X = X.drop(columns=['customerID'])

numeric_columns = X.select_dtypes(include=['number']).columns.tolist()
categorical_columns = [column for column in X.columns if column not in numeric_columns]
print('Numeric:', numeric_columns)
print('Categorical:', categorical_columns)

## Preprocessing pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])

## Train/test split


In [ ]:
stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=stratify)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Baseline Logistic Regression


In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
}
models['LogisticRegression'].fit(X_train_processed, y_train)

## RandomForest / GradientBoosting comparison


In [ ]:
models.update({
    'RandomForestClassifier': RandomForestClassifier(n_estimators=120, class_weight='balanced', random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
})

for name, model in models.items():
    if name != 'LogisticRegression':
        model.fit(X_train_processed, y_train)

## Metrics table


In [ ]:
def evaluate(model, X_eval, y_eval, threshold=0.5):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_eval, predictions),
        'precision': precision_score(y_eval, predictions, zero_division=0),
        'recall': recall_score(y_eval, predictions, zero_division=0),
        'f1': f1_score(y_eval, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_eval, probabilities) if len(np.unique(y_eval)) > 1 else 0.0,
        'confusion_matrix': confusion_matrix(y_eval, predictions).tolist(),
    }

metrics = {name: evaluate(model, X_test_processed, y_test) for name, model in models.items()}
metrics_table = pd.DataFrame(metrics).T.drop(columns=['confusion_matrix'])
metrics_table

## Confusion matrix


In [ ]:
best_name = sorted(metrics, key=lambda name: (metrics[name]['roc_auc'], metrics[name]['recall'], metrics[name]['f1']), reverse=True)[0]
best_model = models[best_name]
predictions = (best_model.predict_proba(X_test_processed)[:, 1] >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
print(best_name, metrics[best_name]['confusion_matrix'])

## ROC-AUC


In [ ]:
RocCurveDisplay.from_predictions(y_test, best_model.predict_proba(X_test_processed)[:, 1])
print('Best ROC-AUC:', metrics[best_name]['roc_auc'])

## Feature importance / explainability


In [ ]:
feature_names = preprocessor.get_feature_names_out()
if hasattr(best_model, 'coef_'):
    importance = best_model.coef_[0]
elif hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
else:
    importance = np.zeros(len(feature_names))

importance_table = pd.DataFrame({
    'feature': feature_names,
    'importance': importance,
    'abs_importance': np.abs(importance),
}).sort_values('abs_importance', ascending=False)
importance_table.head(15)

## Save best model


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_DIR / 'churn_model.joblib')

## Save preprocessing pipeline


In [ ]:
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')
metadata = {
    'model_version': datetime.now(timezone.utc).strftime('%Y.%m.%d.%H%M'),
    'training_date': datetime.now(timezone.utc).isoformat(),
    'model_type': best_name,
    'metrics': {key: (round(value, 4) if isinstance(value, float) else value) for key, value in metrics[best_name].items()},
    'all_model_metrics': metrics,
    'feature_count': len(feature_names),
    'target': TARGET,
    'threshold': 0.5,
    'numeric_columns': numeric_columns,
    'categorical_columns': categorical_columns,
}
with (MODEL_DIR / 'model_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)
metadata

## Limitations


The sample CSV is intentionally small, so its metrics are only smoke-test indicators. A production model requires a larger representative dataset, calibration, drift monitoring, business validation, and careful intervention design.


## Next improvements


Add threshold tuning, probability calibration, SHAP-based visualizations, data validation, batch scoring upload, experiment tracking, and monitoring.
